<a href="https://colab.research.google.com/github/Eboselethefirst/Mini_projects/blob/main/Cleaning%20CsV%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%bash
ls

sample_data


In [44]:
import csv
import pandas as pd
import random
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

In [5]:
num_rows = 100000
file_name= "dirty_data.csv"



In [21]:
def generate_dirty_data():
  date_formats = ("%Y-%m-%d","%d-%m-%Y","NULL","%m-%d-%Y","Invalid-date")
  statuses = ["SUCCESS", "PENDING", "FAILED"]

  with open(file_name, mode="w", encoding='utf-8') as fileone:
    writer = csv.writer(fileone)

    writer.writerow(["transaction_id","user_id","amount","timestamp","status"])

    for i in range(num_rows):
      tx_id = f"TXN-{i}" if random.random() >0.05 else ""
      user_id = random.randint(1000,99999)
      amount = round(random.uniform(1750.0,3500000.0),2)

      fmt = random.choice(date_formats)
      ts = datetime.now().strftime(fmt) if fmt not in ["NULL", "Inavlid-date"] else fmt
      status = random.choice(statuses)
      row = [tx_id, user_id, amount, ts, status]

      #Simulating Schema Drift by adding extra rows

      if random.random() > 0.95 :
        row.extend(["New_feature", "New_feature_2"])

      writer.writerow(row)


    print(f"There are {num_rows} rows in the {file_name} data")

if __name__ == "__main__":
  generate_dirty_data()

There are 100000 rows in the dirty_data.csv data


In [23]:
log_file = open('denied_access.csv', 'a')

def handle_bad_lines(bad_line):
  log_file.write(",".join(bad_line) + "\n")
  return


In [45]:
clean_writer = None

try:
  for chunk in pd.read_csv("dirty_data.csv", chunksize=100000,on_bad_lines=handle_bad_lines, engine='python'):
    #find the ghost rows
    ghost_rows = chunk[chunk["transaction_id"].isna()]

    #Keep only the valid rows
    chunk = chunk.dropna(subset=["transaction_id"])

    #Transform Types
    chunk['amount'] = pd.to_numeric(chunk['amount'], errors = 'coerce')
    chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], errors = 'coerce')

    typo_rows = chunk[chunk['amount'].isna() |  chunk['timestamp'].isna()]

    #Cleaning The data
    clean_chunk = chunk.dropna(subset=['amount', 'timestamp'])

   #Persistence Layer
    if not ghost_rows.empty:
      ghost_rows.to_csv("ghost_transactions.csv", "a", header = False)
    if not typo_rows.empty:
      typo_rows.to_csv("typo_transactions.csv", "a", header = False)

    if not clean_chunk.empty:
      table = pa.Table.from_pandas(clean_chunk)

    if clean_writer is None:
      clean_writer = pq.ParquetWriter("Clean_data.parquet", table.schema)

    clean_writer.write_table(table)

finally:
  log_file.close()
  if clean_writer is not None:
    clean_writer.close()

print("All Files are saved and sorted with all the file Writers Closed ")









/tmp/ipykernel_547/1270427381.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], errors = 'coerce')
/tmp/ipykernel_547/1270427381.py:22: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  ghost_rows.to_csv("ghost_transactions.csv", "a", header = False)
/tmp/ipykernel_547/1270427381.py:24: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  typo_rows.to_csv("typo_transactions.csv", "a", header = False)


All Files are saved and sorted with all the file Writers Closed 


In [46]:
df_new = pd.read_parquet("Clean_data.parquet")

In [47]:
len(df_new)

54137

In [41]:
len(chunk)

54137

In [48]:
%%bash
git init

Initialized empty Git repository in /content/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


In [50]:
%%bash
git branch -m main


In [51]:
%%bash
git add .

In [ ]:
%%bash
git remote add origin https://h